In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("C:/Users/admin/Downloads/ashapura_developers_mulund.xlsx", header=8)
df.head()

,PROJECT_NAME,TIMESTAMP,PM2.5,PM10,TEMPERATURE,HUMIDITY
0,Ashapura Developers,2025-07-08T10:02:11+05:30,17,25,28.9,94.1
1,Ashapura Developers,2025-07-08T10:02:58+05:30,14,26,28.9,94.3
2,Ashapura Developers,2025-07-08T10:03:31+05:30,19,28,28.9,94.0
3,Ashapura Developers,2025-07-08T10:04:01+05:30,19,31,29.0,93.9
4,Ashapura Developers,2025-07-08T10:04:31+05:30,20,26,28.9,94.2


In [2]:
#Converting from timestamp to datetime
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], format='mixed')

In [3]:
#Set Index
df = df.set_index('TIMESTAMP')

In [4]:
# Sort properly
df = df.sort_values('TIMESTAMP')

In [5]:
# Step 4: Remove unrealistic values

df_clean = df.copy()

# Set extreme values to NaN
df_clean.loc[df_clean['PM2.5'] > 500, 'PM2.5'] = np.nan
df_clean.loc[df_clean['PM10'] > 600, 'PM10'] = np.nan

df_clean[['PM2.5','PM10']].describe()

,PM2.5,PM10
count,507871.000000,507843.000000
mean,31.478497,47.937193
std,13.066424,19.898605
min,9.000000,13.000000
25%,20.000000,31.000000
50%,32.000000,46.000000
75%,42.000000,63.000000
max,262.000000,593.000000


In [6]:
# Step 5: Fill missing values smoothly

df_clean['PM2.5'] = df_clean['PM2.5'].interpolate(method='time')
df_clean['PM10'] = df_clean['PM10'].interpolate(method='time')

In [7]:
# Step 6: Apply rolling mean smoothing

df_clean['PM2.5'] = df_clean['PM2.5'].rolling(window=3, min_periods=1).mean()
df_clean['PM10'] = df_clean['PM10'].rolling(window=3, min_periods=1).mean()

In [8]:
# Step 7: Clip to safe realistic ranges

df_clean['PM2.5'] = df_clean['PM2.5'].clip(0, 300)
df_clean['PM10'] = df_clean['PM10'].clip(0, 400)

In [9]:
#Round-Of Data
df_clean[['PM2.5','PM10','TEMPERATURE','HUMIDITY']] = df_clean[['PM2.5','PM10','TEMPERATURE','HUMIDITY']].round(2)

In [10]:
sensor_df = df_clean.copy()

In [11]:
#Resample data into 1hour batch
sensor_df = df.resample('1h').mean(numeric_only=True)

In [12]:
#Checking  Type of Index
print(type(df.index))

<class 'pandas.core.indexes.datetimes.DatetimeIndex'>


In [13]:
#Interpolate: to replace with mode value of any  columns
sensor_df = sensor_df.interpolate(limit_direction='forward')

In [14]:
#Reset Index
sensor_df = sensor_df.reset_index()

In [15]:
sensor_df.head()

,TIMESTAMP,PM2.5,PM10,TEMPERATURE,HUMIDITY
0,2025-07-08 10:00:00+05:30,17.529412,27.529412,28.964706,93.376471
1,2025-07-08 11:00:00+05:30,18.423117,28.791538,29.027348,93.112384
2,2025-07-08 12:00:00+05:30,19.316821,30.053664,29.089990,92.848297
3,2025-07-08 13:00:00+05:30,20.210526,31.315789,29.152632,92.584211
4,2025-07-08 14:00:00+05:30,20.588235,32.201681,29.225210,92.553782


Weather Data

In [16]:
#Weather Data 
weather_df = pd.read_excel("C:/Users/admin/Downloads/weather_data_birwadi_mahad.xlsx", header=16)
weather_df

,From Date,To Date,RH,WS,WD,SR,RF,AT,NO2,Ozone
0,13-08-2025 00:00,13-08-2025 01:00,70.87,2.06,204.50,0.00,0.0,26.19,11.21,14.96
1,13-08-2025 01:00,13-08-2025 02:00,70.82,1.47,234.78,0.00,0.0,26.13,11.04,14.61
2,13-08-2025 02:00,13-08-2025 03:00,70.93,2.18,205.64,0.00,0.0,26.26,11.36,16.14
3,13-08-2025 03:00,13-08-2025 04:00,71.05,1.73,207.66,0.00,0.2,26.39,15.19,14.63
4,13-08-2025 04:00,13-08-2025 05:00,70.90,1.85,207.88,0.00,0.0,26.23,17.10,11.88
...,...,...,...,...,...,...,...,...,...,...
5121,14-03-2026 09:00,14-03-2026 10:00,71.53,1.31,123.37,225.97,0.0,31.08,49.69,49.54
5122,14-03-2026 10:00,14-03-2026 11:00,70.89,1.10,221.14,328.17,0.0,33.91,38.20,51.16
5123,14-03-2026 11:00,14-03-2026 12:00,69.21,1.95,212.03,425.88,0.0,36.07,15.91,50.53
5124,14-03-2026 12:00,14-03-2026 13:00,66.29,2.09,234.81,440.45,0.0,38.21,13.59,49.83


In [17]:
#Remode To Date Columns
weather_df = weather_df.drop('To Date',axis=1)

In [18]:
weather_df.set_index('From Date')

,RH,WS,WD,SR,RF,AT,NO2,Ozone
From Date,,,,,,,,
13-08-2025 00:00,70.87,2.06,204.50,0.00,0.0,26.19,11.21,14.96
13-08-2025 01:00,70.82,1.47,234.78,0.00,0.0,26.13,11.04,14.61
13-08-2025 02:00,70.93,2.18,205.64,0.00,0.0,26.26,11.36,16.14
13-08-2025 03:00,71.05,1.73,207.66,0.00,0.2,26.39,15.19,14.63
13-08-2025 04:00,70.90,1.85,207.88,0.00,0.0,26.23,17.10,11.88
...,...,...,...,...,...,...,...,...
14-03-2026 09:00,71.53,1.31,123.37,225.97,0.0,31.08,49.69,49.54
14-03-2026 10:00,70.89,1.10,221.14,328.17,0.0,33.91,38.20,51.16
14-03-2026 11:00,69.21,1.95,212.03,425.88,0.0,36.07,15.91,50.53


In [19]:
#Replace From Date to TIMESTAMP
weather_df.rename(columns={'From Date':'TIMESTAMP'}, inplace=True)

In [20]:
weather_df = weather_df.set_index('TIMESTAMP')

In [21]:
#Making Timestamp same
sensor_df['TIMESTAMP'] = sensor_df['TIMESTAMP'].dt.tz_localize(None)
#print(sensor_df.columns)
#sensor_df.head()

In [22]:
print(weather_df.columns)

Index(['RH', 'WS', 'WD', 'SR', 'RF', 'AT', 'NO2', 'Ozone'], dtype='object')


In [23]:
weather_df.head()

,RH,WS,WD,SR,RF,AT,NO2,Ozone
TIMESTAMP,,,,,,,,
13-08-2025 00:00,70.87,2.06,204.50,0.0,0.0,26.19,11.21,14.96
13-08-2025 01:00,70.82,1.47,234.78,0.0,0.0,26.13,11.04,14.61
13-08-2025 02:00,70.93,2.18,205.64,0.0,0.0,26.26,11.36,16.14
13-08-2025 03:00,71.05,1.73,207.66,0.0,0.2,26.39,15.19,14.63
13-08-2025 04:00,70.90,1.85,207.88,0.0,0.0,26.23,17.10,11.88


In [24]:
weather_df = weather_df.reset_index()

In [25]:
weather_df['TIMESTAMP'] = pd.to_datetime(
    weather_df['TIMESTAMP'],
    format='%d-%m-%Y %H:%M'
)

In [26]:
weather_df.head()

,TIMESTAMP,RH,WS,WD,SR,RF,AT,NO2,Ozone
0,2025-08-13 00:00:00,70.87,2.06,204.50,0.0,0.0,26.19,11.21,14.96
1,2025-08-13 01:00:00,70.82,1.47,234.78,0.0,0.0,26.13,11.04,14.61
2,2025-08-13 02:00:00,70.93,2.18,205.64,0.0,0.0,26.26,11.36,16.14
3,2025-08-13 03:00:00,71.05,1.73,207.66,0.0,0.2,26.39,15.19,14.63
4,2025-08-13 04:00:00,70.90,1.85,207.88,0.0,0.0,26.23,17.10,11.88


In [27]:
#Merge Data So our ML model can train
merged_df = pd.merge(sensor_df,weather_df, on="TIMESTAMP")

In [29]:
merged_df = merged_df.dropna()

In [31]:
merged_df.size

62426

Model Training - XGBoost

In [32]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

from xgboost import XGBRegressor

In [35]:
merged_df['TIMESTAMP'] = pd.to_datetime(merged_df['TIMESTAMP'], errors='coerce')

merged_df = merged_df.sort_values('TIMESTAMP')

merged_df.set_index('TIMESTAMP', inplace=True)

# FORCE datetime index
merged_df.index = pd.to_datetime(merged_df.index)

In [36]:
# Step 3: Time features

merged_df['hour'] = merged_df.index.hour
merged_df['day'] = merged_df.index.day
merged_df['month'] = merged_df.index.month

In [43]:
# Step 4: Lag features (past values)

for lag in [1, 2, 3, 6, 12, 24, 48, 72]:
    merged_df[f'PM25_lag_{lag}'] = merged_df['PM2.5'].shift(lag)
    merged_df[f'PM10_lag_{lag}'] = merged_df['PM10'].shift(lag)

In [44]:
# Step 5: Rolling features

merged_df['PM25_roll_mean_6'] = merged_df['PM2.5'].rolling(6).mean()
merged_df['PM25_roll_mean_24'] = merged_df['PM2.5'].rolling(24).mean()

merged_df['PM10_roll_mean_6'] = merged_df['PM10'].rolling(6).mean()
merged_df['PM10_roll_mean_24'] = merged_df['PM10'].rolling(24).mean()

In [45]:
# Step 6: Drop NaN (created by lagging)

merged_df = merged_df.dropna()

In [49]:
merged_df.columns

Index(['PM2.5', 'PM10', 'TEMPERATURE', 'HUMIDITY', 'RH', 'WS', 'WD', 'SR',
       'RF', 'AT', 'NO2', 'Ozone', 'hour', 'day', 'month', 'PM25_lag_1',
       'PM10_lag_1', 'PM25_lag_2', 'PM10_lag_2', 'PM25_lag_3', 'PM10_lag_3',
       'PM25_lag_6', 'PM10_lag_6', 'PM25_lag_12', 'PM10_lag_12', 'PM25_lag_24',
       'PM10_lag_24', 'PM25_lag_48', 'PM10_lag_48', 'PM25_lag_72',
       'PM10_lag_72', 'PM25_roll_mean_6', 'PM25_roll_mean_24',
       'PM10_roll_mean_6', 'PM10_roll_mean_24'],
      dtype='object')

In [81]:
# Step 7: Features and targets

features = [
    'TEMPERATURE','HUMIDITY','RH','WS','WD','SR','RF','AT','NO2','Ozone',
    'hour','day','month',
    'PM25_lag_1','PM10_lag_1','PM25_lag_2','PM10_lag_2',
    'PM25_lag_3','PM10_lag_3','PM25_lag_6','PM10_lag_6',
    'PM25_lag_12','PM10_lag_12','PM25_lag_24','PM10_lag_24',
    'PM25_lag_48','PM10_lag_48','PM25_lag_72','PM10_lag_72',
    'PM25_roll_mean_6','PM25_roll_mean_24',
    'PM10_roll_mean_6','PM10_roll_mean_24',
    'hour_sin','hour_cos',
    'PM25_diff','PM10_diff',
    'PM25_roll_std_6','PM10_roll_std_6'
]

In [83]:
#features.remove('PROJECT_NAME')

In [84]:
print(features)

['TEMPERATURE', 'HUMIDITY', 'RH', 'WS', 'WD', 'SR', 'RF', 'AT', 'NO2', 'Ozone', 'hour', 'day', 'month', 'PM25_lag_1', 'PM10_lag_1', 'PM25_lag_2', 'PM10_lag_2', 'PM25_lag_3', 'PM10_lag_3', 'PM25_lag_6', 'PM10_lag_6', 'PM25_lag_12', 'PM10_lag_12', 'PM25_lag_24', 'PM10_lag_24', 'PM25_lag_48', 'PM10_lag_48', 'PM25_lag_72', 'PM10_lag_72', 'PM25_roll_mean_6', 'PM25_roll_mean_24', 'PM10_roll_mean_6', 'PM10_roll_mean_24', 'hour_sin', 'hour_cos', 'PM25_diff', 'PM10_diff', 'PM25_roll_std_6', 'PM10_roll_std_6']


In [85]:
X = merged_df[features]
y_pm25 = merged_df['PM2.5']
y_pm10 = merged_df['PM10']

In [86]:
# Step 8: Split data (time-based)

split = int(len(df) * 0.8)

X_train, X_test = X[:split], X[split:]
y25_train, y25_test = y_pm25[:split], y_pm25[split:]
y10_train, y10_test = y_pm10[:split], y_pm10[split:]

In [87]:
# Step 9: Train models

model_pm25 = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8
)

model_pm10 = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8
)

model_pm25.fit(X_train, y25_train)
model_pm10.fit(X_train, y10_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [88]:
merged_df = merged_df.dropna()

print("After dropna:", merged_df.shape)

After dropna: (4725, 41)


In [89]:
# Ensure correct input
last_row = X.iloc[-1:].copy()

# Match training columns
last_row = last_row[X_train.columns]

# Check shape
print("Input shape:", last_row.shape)

# Predict
pred25 = model_pm25.predict(last_row)[0]
pred10 = model_pm10.predict(last_row)[0]

print("Predicted PM2.5:", pred25)
print("Predicted PM10:", pred10)

Input shape: (1, 39)
Predicted PM2.5: 49.902187
Predicted PM10: 63.994694


In [90]:
# Step 11: Future prediction (iterative)

future_pm25 = []
future_pm10 = []

last_row = X.iloc[-1:].copy()

for i in range(24):
    
    pred25 = model_pm25.predict(last_row)[0]
    pred10 = model_pm10.predict(last_row)[0]
    
    future_pm25.append(pred25)
    future_pm10.append(pred10)
    
    # SHIFT ONLY EXISTING LAGS
    lag_list = [72, 48, 24, 12, 6, 3, 2, 1]
    
    for j in range(len(lag_list)-1, 0, -1):
        curr = lag_list[j]
        prev = lag_list[j-1]
        
        last_row[f'PM25_lag_{curr}'] = last_row[f'PM25_lag_{prev}']
        last_row[f'PM10_lag_{curr}'] = last_row[f'PM10_lag_{prev}']
    
    # update lag_1 with new prediction
    last_row['PM25_lag_1'] = pred25
    last_row['PM10_lag_1'] = pred10

In [91]:
# Step 12: Decision

pm25_next = np.mean(future_pm25)
pm10_next = np.mean(future_pm10)

def decision(pm25, pm10):
    return "CONTINUE WORK" if (pm25 < 100 and pm10 < 150) else "STOP WORK"

print("Tomorrow PM2.5:", pm25_next)
print("Tomorrow PM10:", pm10_next)

print("Decision:", decision(pm25_next, pm10_next))

Tomorrow PM2.5: 47.832428
Tomorrow PM10: 55.143665
Decision: CONTINUE WORK


In [92]:
import joblib

joblib.dump(model_pm25, "xgb_pm25.pkl")
joblib.dump(model_pm10, "xgb_pm10.pkl")

print("✅ Models saved!")

✅ Models saved!


Improve Future Prediction Accuracy

In [93]:
# Time features
merged_df['hour_sin'] = np.sin(2 * np.pi * merged_df.index.hour / 24)
merged_df['hour_cos'] = np.cos(2 * np.pi * merged_df.index.hour / 24)

# Trend features
merged_df['PM25_diff'] = merged_df['PM2.5'].diff()
merged_df['PM10_diff'] = merged_df['PM10'].diff()

In [94]:
merged_df['PM25_roll_std_6'] = merged_df['PM2.5'].rolling(6).std()
merged_df['PM10_roll_std_6'] = merged_df['PM10'].rolling(6).std()

In [95]:
model_pm25 = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8
)

In [96]:
def predict_next_24(model_pm25, model_pm10, X, feature_cols):
    
    future_pm25 = []
    future_pm10 = []
    
    last_row = X.iloc[-1:].copy()

    for i in range(24):
        
        last_row = last_row[feature_cols]

        pred25 = model_pm25.predict(last_row)[0]
        pred10 = model_pm10.predict(last_row)[0]

        future_pm25.append(pred25)
        future_pm10.append(pred10)

        # SHIFT LAGS (SAFE)
        lag_list = [24, 12, 6, 3, 2, 1]

        for j in range(len(lag_list)-1, 0, -1):
            curr = lag_list[j-1]
            prev = lag_list[j]
            
            last_row[f'PM25_lag_{curr}'] = last_row[f'PM25_lag_{prev}']
            last_row[f'PM10_lag_{curr}'] = last_row[f'PM10_lag_{prev}']

        last_row['PM25_lag_1'] = pred25
        last_row['PM10_lag_1'] = pred10

    return future_pm25, future_pm10

In [97]:
merged_df.to_csv("cleaned_ashapura_data.csv")

In [98]:
from xgboost import XGBRegressor
import joblib

model_pm25 = XGBRegressor()
model_pm25.fit(X_train, y25_train)

model_pm10 = XGBRegressor()
model_pm10.fit(X_train, y10_train)

# SAVE MODELS
joblib.dump(model_pm25, "xgb_pm25.pkl")
joblib.dump(model_pm10, "xgb_pm10.pkl")

['xgb_pm10.pkl']

In [99]:
joblib.dump({
    "model_pm25": model_pm25,
    "model_pm10": model_pm10,
    "features": features
}, "model_bundle.pkl")

['model_bundle.pkl']